# Geopolitical Exposure Proof of Concept
## 02B — Expanded Pre-Event Text Corpus

This notebook expands the original 50-firm pilot to a deterministic 300-firm sample without loading event returns. Because the 2022 event results have already been inspected, future same-event V2 analysis is exploratory and must ultimately be confirmed on a separate event.

## 1. Environment and V1 handoff
All expanded artifacts use a separate V2 directory; no V1 file is overwritten.

In [ ]:
from pathlib import Path
import hashlib, importlib, json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name=='geopolitical_exposure': GEO_ROOT,ROOT=ROOT,ROOT.parent
else: GEO_ROOT=ROOT/'geopolitical_exposure'
if not GEO_ROOT.exists(): raise FileNotFoundError('Open from Asset_Embeddings or geopolitical_exposure.')
sys.path.insert(0,str(GEO_ROOT/'src'))
import document_contracts,edgar_pipeline
importlib.reload(document_contracts);importlib.reload(edgar_pipeline)
from document_contracts import normalize_document,validate_documents
from edgar_pipeline import filing_metadata_for_cik,get_text,html_to_text,keyword_passages
V2=GEO_ROOT/'text_expansion_v2';V2_RAW=V2/'raw';V2_PROCESSED=V2/'processed';V2_RESULTS=V2/'results';MANIFESTS=GEO_ROOT/'manifests'
for d in [V2_RAW,V2_PROCESSED,V2_RESULTS,MANIFESTS]: d.mkdir(parents=True,exist_ok=True)
EVENT=json.loads((GEO_ROOT/'config/russia_ukraine_pilot.json').read_text())
CUTOFF=pd.Timestamp(EVENT['information_cutoff']);DOCUMENT_START=pd.Timestamp('2020-01-01')
FORMS=('10-K','10-Q','8-K');TARGET_FIRMS=300;SAMPLE_SEED=20220224;EVENT_OUTCOMES_LOADED=False
v1_freeze=json.loads((MANIFESTS/'text_exposure_seed_freeze_v1.json').read_text())
print('V1 seed hash:',v1_freeze['seed_sha256']);print('Event outcomes loaded:',EVENT_OUTCOMES_LOADED)


## 2. Select and freeze 300 firms
Retain the original 50 firms and select 250 additions across SIC2 industries, size quintiles, and 20 pre-event all-S12 embedding clusters. Event returns do not enter selection.

In [ ]:
from sklearn.cluster import KMeans
selection_path=V2_PROCESSED/'expanded_300_firms_frozen.parquet'
if selection_path.exists():
    expanded_firms=pd.read_parquet(selection_path);print('Loaded frozen selection.')
else:
    bridge=pd.read_parquet(GEO_ROOT/'documents/processed/permco_cik_exact_ticker_bridge.parquet')
    bridge=bridge.query("cik_match_status=='unique_exact_ticker'").drop_duplicates('permco')
    industry=pd.read_parquet(GEO_ROOT/'event_data/crsp_point_in_time_industry_2021q4.parquet')
    controls=pd.read_parquet(ROOT/'remaining_benchmarks/return_comovement/characteristics/2021q4.parquet')
    emb=pd.read_parquet(ROOT/'active_passive_extension/benchmarks/all_s12_control/2021q4/rs_binary/asset_embeddings_10d.parquet')
    dims=[c for c in emb if c.startswith('embedding_')]
    emb=emb[['permco']].assign(embedding_cluster=KMeans(n_clusters=20,random_state=SAMPLE_SEED,n_init=20).fit_predict(emb[dims]))
    eligible=(bridge.merge(industry[['permco','sic2']],on='permco',how='inner',validate='one_to_one')
              .merge(controls[['permco','log_market_equity']],on='permco',how='inner',validate='one_to_one')
              .merge(emb,on='permco',how='inner',validate='one_to_one'))
    eligible['size_quintile']=pd.qcut(eligible.log_market_equity,5,labels=False,duplicates='drop')+1
    pilot_ids=set(pd.read_parquet(GEO_ROOT/'documents/processed/pilot_50_firms_frozen.parquet').permco)
    remaining=eligible[~eligible.permco.isin(pilot_ids)].copy();rng=np.random.default_rng(SAMPLE_SEED)
    remaining['_random']=rng.random(len(remaining));remaining['_round']=remaining.groupby(['sic2','size_quintile','embedding_cluster'])['_random'].rank(method='first')
    additions=remaining.sort_values(['_round','_random']).head(TARGET_FIRMS-len(pilot_ids)).drop(columns=['_random','_round'])
    expanded_firms=pd.concat([eligible[eligible.permco.isin(pilot_ids)],additions],ignore_index=True).drop_duplicates('permco')
    expanded_firms['original_pilot']=expanded_firms.permco.isin(pilot_ids)
    assert len(expanded_firms)==TARGET_FIRMS and expanded_firms.cik.notna().all()
    expanded_firms.to_parquet(selection_path,index=False)
selection_hash=hashlib.sha256(json.dumps(expanded_firms.sort_values('permco').to_dict('records'),sort_keys=True,default=str).encode()).hexdigest()
audit=pd.DataFrame([{'firms':len(expanded_firms),'original_pilot_firms':int(expanded_firms.original_pilot.sum()),'sic2_industries':expanded_firms.sic2.nunique(),'embedding_clusters':expanded_firms.embedding_cluster.nunique(),'size_quintiles':expanded_firms.size_quintile.nunique(),'selection_sha256':selection_hash}])
display(audit)
design={'stage':'expanded_text_corpus_v2','selection_sha256':selection_hash,'v1_seed_sha256':v1_freeze['seed_sha256'],'target_firms':TARGET_FIRMS,'sample_seed':SAMPLE_SEED,'information_cutoff':EVENT['information_cutoff'],'event_outcomes_loaded_in_notebook':False,'same_event_status':'exploratory_after_prior_aggregate_outcome_inspection','confirmation_requirement':'separate held-out geopolitical event'}
design['sha256']=hashlib.sha256(json.dumps(design,sort_keys=True).encode()).hexdigest();(MANIFESTS/'text_corpus_expansion_v2_design.json').write_text(json.dumps(design,indent=2))


## 3. SEC identity and acquisition settings
Enter an identifying SEC user agent containing your university email. All network responses are checkpointed.

In [ ]:
SEC_USER_AGENT=os.getenv('SEC_USER_AGENT','').strip()
if '@' not in SEC_USER_AGENT: SEC_USER_AGENT=input('SEC user agent (example: Academic research name@ubc.ca): ').strip()
if '@' not in SEC_USER_AGENT: raise ValueError('A valid SEC user agent is required.')
print('SEC acquisition configured.')


## 4. Acquire filing metadata
Existing raw CIK submission files from the original pilot are reused. The cell is resumable and prints progress every 25 firms.

In [ ]:
metadata_path=V2_PROCESSED/'expanded_sec_filing_metadata.parquet'
if metadata_path.exists():
    filing_metadata=pd.read_parquet(metadata_path);print('Loaded checkpointed metadata.')
else:
    frames=[];errors=[];submission_cache=GEO_ROOT/'documents/raw/sec_submissions'
    for position,row in enumerate(expanded_firms.itertuples(index=False),1):
        try:
            frame=filing_metadata_for_cik(int(row.cik),SEC_USER_AGENT,FORMS,str(DOCUMENT_START.date()),str(CUTOFF.date()),submission_cache)
            if not frame.empty: frame['permco']=row.permco;frame['crsp_ticker']=row.ticker;frames.append(frame)
        except Exception as error: errors.append({'cik':row.cik,'permco':row.permco,'error_type':type(error).__name__,'error':str(error)})
        if position%25==0: print(f'Metadata {position}/{len(expanded_firms)} complete')
    filing_metadata=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    filing_metadata.to_parquet(metadata_path,index=False);pd.DataFrame(errors).to_parquet(V2_RESULTS/'metadata_errors.parquet',index=False)
metadata_audit=pd.DataFrame([{'sample_firms':len(expanded_firms),'firms_with_filings':filing_metadata.permco.nunique(),'qualifying_filings':len(filing_metadata),'post_cutoff_filings':int((pd.to_datetime(filing_metadata.filingDate)>CUTOFF).sum())}])
display(metadata_audit);display(filing_metadata.groupby('form').size().rename('filings').reset_index())


## 5. Select at most two filings per firm
Choose the latest 10-K and the latest 10-Q filed after it. This deterministic rule limits cost while retaining recent annual and quarterly disclosure.

In [ ]:
selected=[]
for permco,group in filing_metadata.groupby('permco',sort=True):
    group=group.sort_values(['filingDate','accessionNumber']);tens=group[group.form.eq('10-K')]
    if tens.empty: continue
    latest=tens.iloc[-1];selected.append(latest);later=group[group.form.eq('10-Q')&group.filingDate.gt(latest.filingDate)]
    if not later.empty: selected.append(later.iloc[-1])
selected_documents=pd.DataFrame(selected).drop_duplicates('accessionNumber').reset_index(drop=True)
selected_documents.to_parquet(V2_PROCESSED/'expanded_selected_filings.parquet',index=False)
display(selected_documents.groupby('form').size().rename('documents').reset_index());print('Documents:',len(selected_documents),'Firms:',selected_documents.permco.nunique())


## 6. Download and normalize selected filings
Raw HTML is saved one document at a time. This is likely the longest cell and can be safely rerun after interruption.

In [ ]:
normalized_path=V2_PROCESSED/'expanded_sec_documents.jsonl';html_dir=V2_RAW/'sec_filings';html_dir.mkdir(parents=True,exist_ok=True)
normalized_documents=[];errors=[]
for position,row in enumerate(selected_documents.itertuples(index=False),1):
    raw_path=html_dir/f'{row.accessionNumber}.html'
    try:
        html=raw_path.read_text(encoding='utf-8') if raw_path.exists() else get_text(row.source_uri,SEC_USER_AGENT)
        if not raw_path.exists(): raw_path.write_text(html,encoding='utf-8')
        text=html_to_text(html);document_date=pd.to_datetime(row.reportDate,errors='coerce')
        if pd.isna(document_date): document_date=pd.Timestamp(row.filingDate)
        normalized_documents.append(normalize_document({'document_id':row.accessionNumber,'firm_id':str(int(row.permco)),'document_type':row.form,'document_date':document_date.date().isoformat(),'available_date':pd.Timestamp(row.filingDate).date().isoformat(),'source':'SEC EDGAR','section':None,'text':text,'source_uri':row.source_uri}))
    except Exception as error: errors.append({'document_id':row.accessionNumber,'permco':row.permco,'error_type':type(error).__name__,'error':str(error)})
    if position%25==0: print(f'Documents {position}/{len(selected_documents)} complete')
normalized_documents=validate_documents(normalized_documents);normalized_path.write_text(''.join(json.dumps(r)+'\n' for r in normalized_documents))
pd.DataFrame(errors).to_parquet(V2_RESULTS/'document_errors.parquet',index=False)
display(pd.DataFrame([{'documents':len(normalized_documents),'firms':len({r['firm_id'] for r in normalized_documents}),'latest_available_date':max(r['available_date'] for r in normalized_documents),'median_characters':float(np.median([len(r['text']) for r in normalized_documents]))}]))


## 7. Retrieve passages and freeze the API queue
The V1 keywords and passage rules are reused unchanged. This cell does not call OpenAI; it reports the exact queue and approximate token volume for review.

In [ ]:
document_errors=pd.read_parquet(V2_RESULTS/'document_errors.parquet')
if len(document_errors) or len(normalized_documents)!=len(selected_documents):
    raise RuntimeError(f'Corpus incomplete: {len(document_errors)} document errors and {len(selected_documents)-len(normalized_documents)} missing documents. Rerun Section 6 before freezing the queue.')
SEARCH_TERMS=['Russia','Russian','Ukraine','Ukrainian','Crimea','Belarus','Black Sea','Eastern Europe','Commonwealth of Independent States','OFAC','economic sanctions','trade sanctions','export controls','oil and gas production','natural gas production','crude oil production','oil and gas exploration','proved oil reserves','proved gas reserves','Department of Defense','defense contractor','aerospace and defense','military systems','weapons systems','missile systems','fertilizer production','nitrogen fertilizer','potash production','wheat production','grain exports','agricultural exports','maritime shipping','ocean freight','shipping vessels','port operations']
retrieval_records=[]
for document in normalized_documents:
    passages=keyword_passages(document['text'],SEARCH_TERMS,radius=900,max_passages=30)
    retrieval_records.append({'document_id':document['document_id'],'firm_id':document['firm_id'],'document_type':document['document_type'],'available_date':document['available_date'],'content_sha256':document['content_sha256'],'passage_count':len(passages),'retrieved_characters':sum(len(x['text']) for x in passages),'passages':passages})
retrieval_path=V2_PROCESSED/'expanded_retrieval_records.jsonl';retrieval_path.write_text(''.join(json.dumps(r)+'\n' for r in retrieval_records))
retrieval_audit=pd.DataFrame([{k:v for k,v in r.items() if k!='passages'} for r in retrieval_records]);retrieval_audit.to_parquet(V2_RESULTS/'expanded_retrieval_audit.parquet',index=False)
api_queue=retrieval_audit[retrieval_audit.passage_count.gt(0)].copy();api_queue['approx_input_tokens']=np.ceil(api_queue.retrieved_characters/4).astype(int)
queue_hash=hashlib.sha256(json.dumps(api_queue.sort_values('document_id').to_dict('records'),sort_keys=True,default=str).encode()).hexdigest()
queue_manifest={'selection_sha256':selection_hash,'queue_sha256':queue_hash,'documents':len(api_queue),'firms':int(api_queue.firm_id.nunique()),'approx_input_tokens':int(api_queue.approx_input_tokens.sum()),'event_outcomes_loaded':False}
queue_manifest['sha256']=hashlib.sha256(json.dumps(queue_manifest,sort_keys=True).encode()).hexdigest();(MANIFESTS/'text_expansion_v2_api_queue.json').write_text(json.dumps(queue_manifest,indent=2))
display(pd.DataFrame([queue_manifest]));display(retrieval_audit[['passage_count','retrieved_characters']].describe());display(api_queue.sort_values('approx_input_tokens',ascending=False).head(15))
print('Queue frozen and ready for resumable extraction.')


## 8. Freeze the extraction contract and inspect progress
The model, instructions, schema, queue hash, and storage setting are frozen before any API call. Responses are saved individually, so an interruption does not repeat completed calls. Set `RUN_API_BATCH=True` only when ready to process the next batch.


In [ ]:
import importlib,expanded_api_extraction
importlib.reload(expanded_api_extraction)
from expanded_api_extraction import freeze_contract,load_jsonl,run_batch,validate_responses
OPENAI_MODEL='gpt-5.6-luna';API_BATCH_SIZE=25;RUN_API_BATCH=False
schema,extraction_contract,response_dir,remaining_queue,extraction_status=freeze_contract(GEO_ROOT,V2_RESULTS,MANIFESTS,EVENT,selection_hash,queue_hash,api_queue,OPENAI_MODEL,API_BATCH_SIZE)
display(extraction_status)


## 9. Run one resumable API batch
This cell submits at most 25 documents. Every successful response is checkpointed immediately. Rerun Sections 8 and 9 for successive batches; existing responses are skipped automatically.


In [ ]:
if not RUN_API_BATCH:
    print('Batch disabled. Set RUN_API_BATCH=True in Section 8 when ready.')
elif remaining_queue.empty:
    print('All queued documents are already checkpointed.')
else:
    failures=run_batch(remaining_queue,retrieval_records,EVENT,schema,extraction_contract,response_dir)
    failures.to_parquet(V2_RESULTS/'latest_api_batch_failures.parquet',index=False)
    completed=len({p.stem for p in response_dir.glob('*.json')} & set(api_queue.document_id.astype(str)))
    print(f'Checkpointed {completed}/{len(api_queue)} queued documents; {len(failures)} failures in this batch.')


## 10. Validate and consolidate checkpointed responses
Run this after every batch. It checks the frozen contract, source hashes, identifiers, channel membership, score ranges, and exact evidence quotes. Outcome evaluation must wait until all queued documents pass these gates.


In [ ]:
response_audit,exposure_candidates,validation_status=validate_responses(api_queue,normalized_documents,retrieval_records,EVENT,extraction_contract,response_dir,V2_RESULTS)
display(validation_status)
if len(exposure_candidates): display(exposure_candidates.groupby(['channel','direction']).size().rename('exposures').reset_index())
status=validation_status.iloc[0]
if status.remaining_documents>0: print('Continue with another Section 9 batch. Do not begin outcome evaluation yet.')
elif status.response_gate_failures or status.exposure_gate_failures: print('Extraction is complete, but failed validation rows require review.')
else: print('Expanded extraction is complete and all automated validation gates pass.')


## 11. Prepare exact-evidence repair and manual adjudication
Exact model quotations are retained unchanged. For each failed quotation, this section searches only the retrieved filing passages and proposes the closest exact sentence or contiguous sentence block. Proposed repairs remain pending until reviewed; they are never silently accepted.


In [ ]:
importlib.reload(expanded_api_extraction)
from expanded_api_extraction import build_evidence_repair_review
evidence_review,pending_evidence_review=build_evidence_repair_review(exposure_candidates,retrieval_records,V2_RESULTS)
repair_summary=pd.DataFrame([{'candidate_exposures':len(evidence_review),'exact_model_quotes':int(evidence_review.review_decision.eq('accept_exact').sum()),'pending_manual_reviews':len(pending_evidence_review),'median_repair_similarity':pending_evidence_review.repair_similarity.median() if len(pending_evidence_review) else np.nan,'minimum_repair_similarity':pending_evidence_review.repair_similarity.min() if len(pending_evidence_review) else np.nan}])
display(repair_summary)
if len(pending_evidence_review): display(pending_evidence_review[['document_id','firm_id','channel','direction','score','confidence','evidence','proposed_exact_evidence','repair_similarity','review_decision']])
print('Editable review file:',V2_RESULTS/'expanded_evidence_manual_review.csv')


## 12. Finalize the audited expanded exposure panel
This section records the adjudication decisions explicitly. Four supported classifications receive carefully bounded exact source spans. One generic sanctions candidate is rejected for insufficient event specificity. Every accepted quotation must occur in the retrieved filing text and remain within the 1,000-character limit.


In [ ]:
importlib.reload(expanded_api_extraction)
from expanded_api_extraction import finalize_evidence_adjudication
override_specs={
    '0000944809-21-000015':('Pharmsynthez In April 2013','development activities.'),
    '0001564590-21-009169':('The Company’s major clients are','Department of Defense.'),
    '0001564590-22-005714':('An economic downturn, as well as','among other factors .'),
    '0001681622-22-000011':('Our products are sold in three geographic regions','Revenues by region are based on the known final destination of products sold.'),
}
rejected_document_ids={'0001730168-21-000153'}
adjudication_audit,final_exposure_panel=finalize_evidence_adjudication(evidence_review,retrieval_records,override_specs,rejected_document_ids,V2_RESULTS)
decision_summary=adjudication_audit.groupby('final_decision').size().rename('candidates').reset_index()
final_summary=pd.DataFrame([{'model_candidates':len(adjudication_audit),'accepted_candidates':len(final_exposure_panel),'rejected_candidates':int((~adjudication_audit.accepted).sum()),'accepted_firms':final_exposure_panel.firm_id.nunique(),'accepted_documents':final_exposure_panel.document_id.nunique(),'all_final_quotes_valid':bool(adjudication_audit.final_quote_valid.all())}])
display(final_summary);display(decision_summary)
display(final_exposure_panel.groupby(['channel','direction']).size().rename('accepted_exposures').reset_index())
print('Final audited exposure panel:',V2_RESULTS/'expanded_exposure_panel_final.parquet')
